# E7 — Best Combo

**E7 — GloVe + LSTM + Dropout + Early Stopping + LR Scheduling + Class Weights.**

In [ ]:
import os, re, time, json, pickle
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = "../data/IMDB Dataset.csv"
RESULTS_DIR = "../results"
TOKENIZER_PATH = "../results/tokenizer.pkl"
VOCAB_SIZE = 10000
EMBED_DIM = 100
SAMPLE_SIZE = 15000   # <-- subsample for faster training

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["review"] = df["review"].apply(lambda t: re.sub(r"<br\s*/?>", " ", str(t)))
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
assert df["label"].isna().sum() == 0, "Unexpected sentiment values — check the column."

# Subsample BEFORE splitting, so train/test shrink together and stay balanced
df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

X = df["review"].astype(str).to_numpy()
y = df["label"].to_numpy(dtype=int)

from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(SEED)
np.random.seed(SEED)
MAX_LEN = 200
BATCH_SIZE = 64
EPOCHS = 15
DROPOUT = 0.3
RECURRENT_DROPOUT = 0.2

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Reuse the SAME tokenizer across every notebook (loads from disk if a
# previous notebook already built one) so all experiments share one vocab —
# that's what makes comparing accuracy across them fair.
if os.path.exists(TOKENIZER_PATH):
    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)
    print("Loaded existing tokenizer.")
else:
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("Built and saved new tokenizer.")

In [ ]:
GLOVE_PATH = "../data/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    raise FileNotFoundError(
        f"{GLOVE_PATH} not found. Download glove.6B.zip from "
        "https://nlp.stanford.edu/projects/glove/ and unzip into data/."
    )

embeddings_index = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        values = line.split()
        embeddings_index[values[0]] = np.asarray(values[1:], dtype="float32")

embedding_matrix = np.random.normal(scale=0.1, size=(VOCAB_SIZE, EMBED_DIM)).astype("float32")
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    if word in embeddings_index:
        embedding_matrix[idx] = embeddings_index[word]
        hits += 1
print(f"GloVe coverage: {hits}/{VOCAB_SIZE} ({hits/VOCAB_SIZE:.1%})")

In [ ]:
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(weights))
print("Class weights:", class_weight_dict)

x_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
x_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
    LSTM(64, dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT),
    Dropout(DROPOUT),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
]

start = time.time()
history = model.fit(x_train, y_train, validation_split=0.1,
                     batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2,
                     callbacks=callbacks, class_weight=class_weight_dict)
train_time = time.time() - start

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

probs = model.predict(x_test, batch_size=BATCH_SIZE).ravel()
preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary")
auc = roc_auc_score(y_test, probs)

row = {
    "run_name": "E7_best_combo",
    "accuracy": round(acc, 4), "precision": round(prec, 4), "recall": round(rec, 4),
    "f1": round(f1, 4), "roc_auc": round(auc, 4),
    "train_time_sec": round(train_time, 1),
    "epochs_run": len(history.history["loss"]),
    "params": model.count_params(),
    "max_len": MAX_LEN, "embeddings": "glove", "dropout": DROPOUT, "lr_schedule": True, "early_stopping": True, "class_weights": True,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "E7_best_combo.csv")
pd.DataFrame([row]).to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(json.dumps(row, indent=2))